# Update Website Inventory from Real Car Records

This notebook locates the real car inventory stored under `src/data/Cars Inventory`, extracts metadata from the `.docx` inventory files, and prepares the data in the website's expected vehicle model.

In [18]:
import json
import re
import shutil
import zipfile
from pathlib import Path
from typing import Dict, List, Optional

import docx

inventory_dir = Path("src/data/Cars Inventory")
assert inventory_dir.exists(), f"Inventory folder not found: {inventory_dir}"

## Locate and Load Car Inventory Data

Find each car inventory folder and load the `.docx` metadata files inside.

In [19]:
def extract_docx_text(path: Path) -> str:
    try:
        doc = docx.Document(path)
        text = ' '.join(p.text for p in doc.paragraphs if p.text)
    except Exception:
        try:
            with zipfile.ZipFile(path) as z:
                xml = z.read('word/document.xml').decode('utf-8')
            text = re.sub(r'</w:t>\s*<w:t[^>]*>', '', xml)
            text = re.sub(r'<[^>]+>', '', text)
        except Exception:
            return ''
    text = text.replace('\n', ' ').strip()
    return re.sub(r'\s+', ' ', text)

car_docs = []
corrupt_files = []
for folder in sorted(inventory_dir.iterdir()):
    if folder.is_dir():
        docx_files = list(folder.glob('*.docx'))
        if not docx_files:
            continue
        file = docx_files[0]
        text = extract_docx_text(file)
        if not text:
            corrupt_files.append((folder.name, file.name))
            continue
        car_docs.append((folder.name, file, text))

print('Vehicles parsed:', len(car_docs))
print('Corrupt/unsupported files:', corrupt_files)
print('Sample vehicles:', [name for name, _, _ in car_docs][:10])

Vehicles parsed: 16
Corrupt/unsupported files: [('2018 SM5 NOVA', '2018 SM5 NOVA.docx')]
Sample vehicles: ['2013 Kia GrandBird', '2015 Kia Moring 4', '2016 Kia Morning', '2016 Kia Morning 2', '2016 Kia Morning 3', '2016 Kia Morning 5', '2016 Kia Morning 6', '2016 Sportage 4th Gen', '2017 Accent', '2017 Chevrolet Spark']


## Inspect and Clean Car Records

Preview the extracted text for the first few vehicles and identify the fields we can use for the website data model.

In [8]:
for name, docx_path, text in car_docs:
    print('---')
    print(name)
    print(text)


---
2013 Kia GrandBird
2013 KIA GRANDBIRD SILK ROAD 47-SEATER DIESEL MANUAL FULL ACCESSORIES $18,452Shipment by RORO $17,100
---
2015 Kia Moring 4
2015 Kia Morning 1.0 Gasoline Key 🔑 Reverse Camera$3800
---
2016 Kia Morning
2016 Kia Morning1.0 Gasoline Smart KeyReverse Camera $4500
---
2016 Kia Morning 2
2016 Kia Morning 1.0 Gasoline Smart KeyReverse CameraCruise Control$4400
---
2016 Kia Morning 3
2016 Kia Morning 1.0 Gasoline Key 🔑 Reverse Camera$3800
---
2016 Kia Morning 5
2016 model Kia Morning 1.0 GasolineKey🔑 Cruise Control$4000
---
2016 Kia Morning 6
2016 Kia Morning 1.0 Gasoline Key 🔑 Reverse Camera$3500
---
2016 Sportage 4th Gen
2016 Sportage 4th Gen *NO ACCIDENT *Mileage: 125,987km1.7 DIESEL GK144943SMART KEYNAVIGATION CAMERA FULL AUTO ACCRUISE CONTROL HEATING SEAT PRICE: $7300
---
2017 Accent
2017 Accent1.6 cc DIESEL HU291675SMART KEYNAVIGATION CAMERA HEATING SEAT$5,000
---
2017 Chevrolet Spark
2017 Chevrolet Spark, 1.0 Gasoline$4500
---
2017 Range Rover Evoque
2017 Range Ro

## Map Inventory to Website Data Model

Build a normalized vehicle dictionary that matches the app's `Vehicle` type.

In [23]:
def slugify(text: str) -> str:
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')

brand_aliases = {
    'accent': 'Hyundai',
    'avante': 'Hyundai',
    'tucson': 'Hyundai',
    'sm5': 'Samsung',
    'k3': 'Kia',
    'seltos': 'Kia',
    'sportage': 'Kia',
    'morning': 'Kia',
    'moring': 'Kia',
    'grandbird': 'Kia',
    'spark': 'Chevrolet',
    'range rover': 'Land Rover',
}

category_map = {
    'accent': 'Sedan',
    'avante': 'Sedan',
    'sm5': 'Sedan',
    'k3': 'Sedan',
    'grandbird': 'Van',
    'spark': 'Hatchback',
    'morning': 'Hatchback',
    'moring': 'Hatchback',
    'sportage': 'SUV',
    'tucson': 'SUV',
    'seltos': 'SUV',
    'range rover': 'Luxury SUV',
}

def parse_price(text: str) -> int:
    m = re.search(r'\$\s*([\d,]+)', text)
    if m:
        return int(m.group(1).replace(',', ''))
    return 0


def parse_mileage(text: str) -> int:
    m = re.search(r'Mileage[:\s]*([\d,]+)km', text, re.IGNORECASE)
    if m:
        return int(m.group(1).replace(',', ''))
    return 0


def parse_engine_cc(text: str) -> int:
    m = re.search(r'([0-9]+\.[0-9]+)\s*cc', text, re.IGNORECASE)
    if m:
        return int(float(m.group(1)) * 1000)
    m = re.search(r'([0-9]+\.[0-9]+)\s*(gasoline|diesel|hybrid|lpg)', text, re.IGNORECASE)
    if m:
        return int(float(m.group(1)) * 1000)
    return 0


def parse_fuel(text: str) -> str:
    for fuel in ['Hybrid', 'Diesel', 'Gasoline', 'LPG']:
        if fuel.lower() in text.lower():
            return fuel
    return 'Unknown'


def parse_transmission(text: str) -> str:
    return 'Manual' if 'manual' in text.lower() else 'Automatic'


def parse_features(text: str) -> List[str]:
    features = []
    keywords = [
        'Smart Key', 'Reverse Camera', 'Navigation', 'Cruise Control',
        'Heating Seat', 'Vent Seat', 'BSD', 'LDWS', 'LDW', 'Camera',
        'Panorama Sunroof', 'Full Auto', 'Power Seat', 'Power Trunk',
        '4WD', 'No Accident', 'Full Accessories', 'Smart Key', 'Key',
    ]
    lower = text.lower()
    for kw in keywords:
        if kw.lower() in lower and kw not in features:
            features.append(kw)
    return features


def choose_brand_and_model(folder_name: str):
    parts = folder_name.split(' ', 1)
    year = int(parts[0])
    rest = parts[1] if len(parts) > 1 else ''
    lower = rest.lower()
    brand = 'Unknown'
    for token, alias in brand_aliases.items():
        if token in lower:
            brand = alias
            break
    model = rest
    if brand != 'Unknown' and lower.startswith(brand.lower()):
        model = rest[len(brand):].strip()
    if model == '':
        model = rest
    return year, brand, model


def choose_category(folder_name: str) -> str:
    lower = folder_name.lower()
    for token, category in category_map.items():
        if token in lower:
            return category
    return 'Sedan'


def choose_drivetrain(folder_name: str) -> str:
    lower = folder_name.lower()
    if 'range rover' in lower:
        return 'AWD'
    if 'sportage' in lower or 'tucson' in lower or 'seltos' in lower:
        return 'FWD'
    if 'accent' in lower or 'avante' in lower or 'k3' in lower or 'spark' in lower or 'morning' in lower or 'moring' in lower:
        return 'FWD'
    if 'grandbird' in lower:
        return 'RWD'
    return 'FWD'


def choose_doors(category: str) -> int:
    return 5 if category in ['SUV', 'Hatchback', 'Luxury SUV', 'Van'] else 4


def choose_seats(category: str) -> int:
    return 47 if category == 'Van' else 5


def make_vehicle_record(folder_name: str, text: str, imgs: List[str]) -> Dict:
    year, brand, model = choose_brand_and_model(folder_name)
    category = choose_category(folder_name)
    price = parse_price(text)
    mileage = parse_mileage(text)
    engine_cc = parse_engine_cc(text)
    fuel_type = parse_fuel(text)
    transmission = parse_transmission(text)
    drivetrain = choose_drivetrain(folder_name)
    features = parse_features(text)
    return {
        'id': len(parsed) + 1,
        'brand': brand,
        'model': model,
        'year': year,
        'trim': '',
        'mileage': mileage,
        'fuelType': fuel_type,
        'transmission': transmission,
        'drivetrain': drivetrain,
        'category': category,
        'price': price,
        'status': 'Available',
        'engineCC': engine_cc,
        'horsepower': 0,
        'torque': 'N/A',
        'fuelEconomy': 'N/A',
        'seats': choose_seats(category),
        'color': 'Unknown',
        'doors': choose_doors(category),
        'features': features,
        'auctionGrade': 'N/A',
        'description': text,
        'img': imgs[0] if imgs else '',
        'imgs': imgs,
        'views': 0,
        'inquiries': 0,
    }

parsed = []
for name, docx_path, text in car_docs:
    slug = slugify(name)
    folder = inventory_dir / name
    image_files = sorted([p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
    imgs = []
    for idx, image in enumerate(image_files[:12], start=1):
        out_dir = Path('public/inventory') / slug
        out_dir.mkdir(parents=True, exist_ok=True)
        dest = out_dir / f'{idx}{image.suffix.lower()}'
        shutil.copyfile(image, dest)
        imgs.append(f'/inventory/{slug}/{dest.name}')
    record = make_vehicle_record(name, text, imgs)
    parsed.append(record)

print('parsed count:', len(parsed))
print('first 3 ids and models:', [(v['id'], v['brand'], v['model'], v['price']) for v in parsed[:3]])
print('missing details:', [v['brand'] + ' ' + v['model'] for v in parsed if v['price'] == 0])

parsed count: 16
first 3 ids and models: [(1, 'Kia', 'GrandBird', 18452), (2, 'Kia', 'Moring 4', 3800), (3, 'Kia', 'Morning', 4500)]
missing details: []


## Write Updated Website Data and Preview

Convert the parsed records into the app's inventory file format and write a new TypeScript data file.

In [24]:
def ts_value(value):
    if isinstance(value, str):
        return json.dumps(value)
    if isinstance(value, list):
        return '[' + ', '.join(ts_value(v) for v in value) + ']'
    return str(value)


def object_to_ts(obj):
    lines = []
    for key, value in obj.items():
        lines.append(f"  {key}: {ts_value(value)},")
    return '{\n' + '\n'.join(lines) + '\n}'

output_path = Path('src/data/vehicles.ts')
content = 'import type { Vehicle } from "@/types/vehicle";\n\n'
content += 'export const vehicles: Vehicle[] = [\n'
for v in parsed:
    content += object_to_ts(v) + ',\n'
content += '];\n'
output_path.write_text(content, encoding='utf-8')
print('Written', output_path)
print('Vehicle count:', len(parsed))
print('Example:', parsed[0])

Written src\data\vehicles.ts
Vehicle count: 16
Example: {'id': 1, 'brand': 'Kia', 'model': 'GrandBird', 'year': 2013, 'trim': '', 'mileage': 0, 'fuelType': 'Diesel', 'transmission': 'Manual', 'drivetrain': 'RWD', 'category': 'Van', 'price': 18452, 'status': 'Available', 'engineCC': 0, 'horsepower': 0, 'torque': 'N/A', 'fuelEconomy': 'N/A', 'seats': 47, 'color': 'Unknown', 'doors': 5, 'features': ['Full Accessories'], 'auctionGrade': 'N/A', 'description': '2013 KIA GRANDBIRD SILK ROAD 47-SEATER DIESEL MANUAL FULL ACCESSORIES $18,452 Shipment by RORO $17,100', 'img': '/inventory/2013-kia-grandbird/1.jpeg', 'imgs': ['/inventory/2013-kia-grandbird/1.jpeg', '/inventory/2013-kia-grandbird/2.jpeg', '/inventory/2013-kia-grandbird/3.jpeg', '/inventory/2013-kia-grandbird/4.jpeg', '/inventory/2013-kia-grandbird/5.jpeg', '/inventory/2013-kia-grandbird/6.jpeg', '/inventory/2013-kia-grandbird/7.jpeg', '/inventory/2013-kia-grandbird/8.jpeg', '/inventory/2013-kia-grandbird/9.jpeg', '/inventory/2013-ki